# Assignment


## Brief

Write the Python codes for the following questions.


## Instructions


Paste the answer as Python in the answer code section below each question. Make sure the Kafka docker is running and you also run the producer notebook to generate orders.


In [59]:
import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.4 pyspark-shell'

# set the options to connect to our Kafka cluster
options = {
    # "kafka.sasl.jaas.config": 'org.apache.kafka.common.security.scram.ScramLoginModule required username="YnJhdmUtZmlzaC0xMTQ2MyQSvwXBuLOQsV1W7YffuC8cDaZcA3fKQwakMhnQGgg" password="MDUxNjc4YzEtYzYxNy00NTE1LWEwNWYtMDBhODRlZmE0OGJm";',
    # "kafka.sasl.mechanism": "SCRAM-SHA-256",
    # "kafka.security.protocol" : "SASL_SSL",
    "kafka.bootstrap.servers": 'localhost:9092',
    "subscribe": 'pizza-orders',
}

In [60]:
# We will only use 2 cores below to speed up creation of the spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("LearnSparkStreaming").master("local[2]").getOrCreate()

In [61]:
spark

### Question 1

Calculate the average no. of pizzas (count) by shop.


In [62]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StringType, IntegerType, LongType, DoubleType, StructType, ArrayType, StructField

pizza_df = spark.readStream.format('kafka')\
    .options(**options)\
    .load()

pizza_schema = StructType([
  StructField("pizzaName", StringType()),
  StructField("additionalToppings", ArrayType(StringType())),
])

order_schema = StructType([
  StructField("address", StringType()),
  StructField("id", IntegerType()),
  StructField("name", StringType()),
  StructField("phoneNumber", StringType()),
  StructField("shop", StringType()),
  StructField("cost", DoubleType()),
  StructField("pizzas", ArrayType(pizza_schema)),
  StructField("timestamp", LongType()),
])

parsed_df = pizza_df.select("timestamp", from_json(col("value").cast("string"), order_schema).alias("value"))

**Answer:**

In [63]:
parsed_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- address: string (nullable = true)
 |    |-- id: integer (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- phoneNumber: string (nullable = true)
 |    |-- shop: string (nullable = true)
 |    |-- cost: double (nullable = true)
 |    |-- pizzas: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- pizzaName: string (nullable = true)
 |    |    |    |-- additionalToppings: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |-- timestamp: long (nullable = true)



In [64]:
print(parsed_df.isStreaming) 

True


In [65]:
## Complete example with counting
from pyspark.sql import functions as F
from pyspark.sql.functions import size
ALL_SHOPS = ["Marios Pizza",
            "Luigis Pizza",
            "Circular Pi Pizzeria",
            "Ill Make You a Pizza You Can" "t Refuse",
            "Mammamia Pizza",
            "Its-a me! Mario Pizza!"]

accumulated = {shop: {"total_pizzas": 0, "total_orders": 0} for shop in ALL_SHOPS}

def process_batch(batch_df, batch_id):
    global accumulated
    
    # get per-shop stats from this batch
    batch_stats = batch_df.groupBy("value.shop").agg(
        F.sum(size("value.pizzas")).alias("total_pizzas"),
        F.count("*").alias("total_orders")
    ).collect()
    
    # accumulate across batches
    for row in batch_stats:
        shop = row["shop"]
        if shop not in accumulated:
            accumulated[shop] = {"total_pizzas": 0, "total_orders": 0}
        accumulated[shop]["total_pizzas"] += row["total_pizzas"]
        accumulated[shop]["total_orders"] += row["total_orders"]
    
    # print accumulated averages
    print("+----------------+---------------+---------------+-----------------+----------------+")
    print("|                                shop|total_orders |total_pizzas |avg_pizza_count|")
    print("+----------------+---------------+---------------+-----------------+----------------+")
    for shop, data in accumulated.items():
        total_orders = data["total_orders"]
        total_pizzas = data["total_pizzas"]
        avg = total_pizzas / total_orders if total_orders > 0 else 0 
        print(f"|{shop:>36}|{total_orders:>13}|{total_pizzas:>13}|{avg:>12.2f}|")
    print("+----------------+---------------+---------------+-----------------+-----------------+")

query = parsed_df \
    .writeStream \
    .trigger(processingTime="60 seconds") \
    .foreachBatch(process_batch) \
    .start()

query.awaitTermination()

26/06/20 19:57:57 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d54b731d-f511-436c-9b1e-f3e7afca85ae. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/20 19:57:57 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/20 19:57:58 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+----------------+---------------+---------------+-----------------+----------------+
|                                shop|total_orders |total_pizzas |avg_pizza_count|
+----------------+---------------+---------------+-----------------+----------------+
|                        Marios Pizza|            0|            0|        0.00|
|                        Luigis Pizza|            0|            0|        0.00|
|                Circular Pi Pizzeria|            0|            0|        0.00|
|Ill Make You a Pizza You Cant Refuse|            0|            0|        0.00|
|                      Mammamia Pizza|            0|            0|        0.00|
|              Its-a me! Mario Pizza!|            0|            0|        0.00|
+----------------+---------------+---------------+-----------------+-----------------+


+----------------+---------------+---------------+-----------------+
|                               shop|total_orders |total_pizzas |    avg_pizza_count|
+----------------+---------------+---------------+-----------------+
|                        Marios Pizza|             0|             0|        0.00|
|                        Luigis Pizza|             0|             0|        0.00|
|                Circular Pi Pizzeria|             0|             0|        0.00|
|Ill Make You a Pizza You Cant Refuse|             1|             5|        5.00|
|                      Mammamia Pizza|             0|             0|        0.00|
|              Its-a me! Mario Pizza!|             0|             0|        0.00|
+----------------+---------------+---------------+-----------------+
+----------------+---------------+---------------+-----------------+
|                     shop|total_orders |total_pizzas |    avg_pizza_count|
+----------------+---------------+---------------+-----------------+
|

+----------------+---------------+---------------+-----------------+
|                     shop|total_orders |total_pizzas |    avg_pizza_count|
+----------------+---------------+---------------+-----------------+
|                        Marios Pizza|             1|             1|           1.00|
|                        Luigis Pizza|             2|             4|           2.00|
|                Circular Pi Pizzeria|             0|             0|           0.00|
|Ill Make You a Pizza You Cant Refuse|             6|            30|           5.00|
|                      Mammamia Pizza|             0|             0|           0.00|
|              Its-a me! Mario Pizza!|             1|             3|           3.00|
+----------------+---------------+---------------+-----------------+
+----------------+---------------+---------------+-----------------+
|                               shop|total_orders |total_pizzas |    avg_pizza_count|
+----------------+---------------+---------------+--

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/liuyx/miniconda3/envs/kafka/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/liuyx/miniconda3/envs/kafka/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/liuyx/miniconda3/envs/kafka/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
query.stop()

+----------------+---------------+---------------+-----------------+
|                                shop|total_orders |total_pizzas |    avg_pizza_count|
+----------------+---------------+---------------+-----------------+
|                        Marios Pizza|             7|             7|        1.00|
|                        Luigis Pizza|            14|            28|        2.00|
|                Circular Pi Pizzeria|             0|             0|        0.00|
|Ill Make You a Pizza You Cant Refuse|             7|            35|        5.00|
|                      Mammamia Pizza|             0|             0|        0.00|
|              Its-a me! Mario Pizza!|             7|            21|        3.00|
+----------------+---------------+---------------+-----------------+
